In [30]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("./data/train_cleaned.csv")
print(f"Loaded {len(df):,} rows")

%load_ext autoreload
%autoreload 2

Loaded 891 rows
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
X = df[['Pclass', 'Sex', 'Age', 'Fare']]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
# Scaling features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Converting to tensors, y must have the shape [N, 1] and [N]
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [33]:
model = nn.Sequential(
    nn.Linear(in_features=4, out_features=1),
    nn.Sigmoid(),
)

prob = model(X_train_tensor)
print(prob[:5])

tensor([[0.6956],
        [0.5996],
        [0.7076],
        [0.6683],
        [0.3826]], grad_fn=<SliceBackward0>)


In [34]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(1000):
    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = loss_fn(output, y_train_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d}, loss {loss.item():.4f}')

Epoch    0, loss 0.9177
Epoch  100, loss 0.4603
Epoch  200, loss 0.4603
Epoch  300, loss 0.4603
Epoch  400, loss 0.4603
Epoch  500, loss 0.4603
Epoch  600, loss 0.4603
Epoch  700, loss 0.4603
Epoch  800, loss 0.4603
Epoch  900, loss 0.4603


In [35]:
with torch.no_grad():
    pred = model(X_test_tensor)
    rounded_pred = (pred > 0.5).float()

    matches = rounded_pred == y_test_tensor

    correct_count = matches.sum().item()
    total_count = len(y_test_tensor)

    acc = correct_count / total_count
    print(f"Accuracy: {acc * 100:.2f}%")


Accuracy: 79.89%
